# NavPlotter: addestramento del modello marino

Questo notebook addestra un YOLOv8s che riconosce **barche, navi, barche a vela, persone in acqua, boe e ostacoli
galleggianti**, partendo dal modello COCO e dai dataset marini che gli dai. Alla fine scarichi uno zip con
`best.onnx`, le immagini di calibrazione e `classes.json`: lo compili per l'Hailo con `training/compile.sh`
(vedi il README della cartella `training`).

Prima di tutto: **Runtime → Cambia tipo di runtime → GPU** (la T4 gratuita va bene). Poi esegui le celle in ordine.

In [12]:
!nvidia-smi -L
!pip -q install ultralytics roboflow
!git clone -q https://github.com/digitalimpresys/Navplotter-Linux.git
%cd /content

GPU 0: Tesla T4 (UUID: GPU-3c0e8fe9-d9e5-76af-99f9-4ec7f6508eda)
fatal: destination path 'Navplotter-Linux' already exists and is not an empty directory.
/content


## 1. I dati

Metti tutto in `/content/inputs`, una cartella (o un file COCO `.json`) per ogni dataset:

- **I tuoi filmati**: dalla pagina Telecamera scarichi lo zip di una registrazione; etichettalo (Roboflow, CVAT o
  Label Studio, esporta in formato YOLO) e caricalo qui o su Google Drive.
- **Roboflow Universe** (universe.roboflow.com): cerca per esempio *buoy*, *boat detection*, *marine obstacle*,
  *man overboard*, *sea drone*. Dalla pagina del dataset, *Download → YOLOv8 → show download code* ti dà le righe
  da incollare nella cella qui sotto (la chiave API è la tua, gratuita).
- **SeaDronesSee** (seadronessee.cs.uni-tuebingen.de, *Object Detection v2*), **LaRS** e **MODS**: scaricali dal loro sito,
  scompattali in `/content/inputs` e indica il loro `.json` COCO.

Le classi dei dataset vengono tradotte nelle nostre con i sinonimi di `training/classes.json`; quelle che non
conosce le scarta e te le elenca, così puoi aggiungerle ai sinonimi se servono.

In [13]:
# Facoltativo: i file da Google Drive (per esempio le tue registrazioni etichettate, come zip)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/inputs
# !unzip -q -o "/content/drive/MyDrive/navplotter/registrazione-etichettata.zip" -d /content/inputs/mie-registrazioni

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# Facoltativo: dataset di Roboflow Universe. Incolla qui le righe di "show download code" e ripeti per ogni dataset.
# from roboflow import Roboflow
# rf = Roboflow(api_key="LA_TUA_CHIAVE")
# rf.workspace("...").project("...").version(1).download("yolov8", location="/content/inputs/boe")

In [15]:
import glob, os
inputs = sorted(glob.glob('/content/inputs/*/'))
# I dataset COCO: aggiungi qui i loro .json, per esempio
# inputs += ['/content/inputs/seadronessee/annotations/instances_train.json']
print('\n'.join(inputs))
!python3 Navplotter-Linux/training/merge_datasets.py -o /content/dataset {' '.join(inputs)}


usage: merge_datasets.py [-h] [-o OUT] [--classes CLASSES] [--val VAL]
                         [--images IMAGES] [--no-empty]
                         inputs [inputs ...]
merge_datasets.py: error: the following arguments are required: inputs


## 2. L'addestramento

Con qualche migliaio di immagini sulla T4 ci vuole un'ora o due. `patience` lo ferma da solo quando smette di
migliorare. Se Colab ti scollega, riesegui la cella con `resume=True` sul `last.pt`.

In [17]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')           # yolov8s: è quello che il Model Zoo di Hailo sa compilare con lo script
model.train(data='/content/dataset/data.yaml', imgsz=640, epochs=100, patience=20, batch=16,
            project='/content/runs', name='marine')
# Per riprendere: YOLO('/content/runs/marine/weights/last.pt').train(resume=True)

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=marine-4, nbs=64, nms=None, opset=None, optimize=False, optimi

RuntimeError: Dataset '/content/dataset/data.yaml' error ❌ '/content/dataset/data.yaml' does not exist

In [ ]:
# Come se la cava sulle immagini di validazione: mAP50 per classe
metrics = YOLO('/content/runs/marine/weights/best.pt').val(data='/content/dataset/data.yaml')
for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f'{metrics.names[int(i)]:10s} {ap:.2f}')

In [ ]:
# Una prova a occhio su qualche immagine di validazione
import glob
from IPython.display import Image, display
res = YOLO('/content/runs/marine/weights/best.pt').predict(sorted(glob.glob('/content/dataset/images/val/*'))[:6], conf=0.3)
for r in res:
    r.save(filename='/content/pred.jpg'); display(Image('/content/pred.jpg', width=640))

## 3. Esportazione per l'Hailo

L'ONNX va compilato per l'Hailo-8 su un PC Linux x86 con la Hailo AI Software Suite (vedi README). Lo zip
contiene anche 512 immagini del dataset per la calibrazione.

In [ ]:
import random, shutil, os, glob
YOLO('/content/runs/marine/weights/best.pt').export(format='onnx', imgsz=640, opset=11)
out = '/content/navplotter-marine'
shutil.rmtree(out, ignore_errors=True); os.makedirs(out + '/calib')
shutil.copy('/content/runs/marine/weights/best.onnx', out)
shutil.copy('/content/runs/marine/weights/best.pt', out)
shutil.copy('/content/dataset/classes.json', out)
imgs = sorted(glob.glob('/content/dataset/images/train/*'))
random.seed(0)
for p in random.sample(imgs, min(512, len(imgs))):
    shutil.copy(p, out + '/calib')
shutil.make_archive('/content/navplotter-marine', 'zip', out)
from google.colab import files
files.download('/content/navplotter-marine.zip')